In [ ]:
import pandas as pd
import ast

csv_path = r"C:/Sebi/Master/code/Simple-Test-LLMs-Politifact/results/zero-shot-lac/crime/conformal_results_all_trials_100trials_20260327_141313.csv"
df = pd.read_csv(csv_path)

true_side_labels = {"true", "mostly-true", "half-true"}
false_side_labels = {"mostly-false", "false", "pants-fire"}

def verdict_side(verdict):
    if verdict in true_side_labels:
        return "true"
    if verdict in false_side_labels:
        return "false"
    return "unknown"

def parse_prediction_set(value):
    if isinstance(value, str):
        return ast.literal_eval(value)
    return value

def prediction_set_side(prediction_set):
    labels = parse_prediction_set(prediction_set)
    true_count = sum(label in true_side_labels for label in labels)
    false_count = sum(label in false_side_labels for label in labels)

    if true_count > false_count:
        return "true"
    if false_count > true_count:
        return "false"
    return "neutral"

df["true_side"] = df["true_verdict"].apply(verdict_side)
df["prediction_side"] = df["prediction_set"].apply(prediction_set_side)

df["side_matches"] = df["prediction_side"] == df["true_side"]
df["is_neutral"] = df["prediction_side"] == "neutral"

total_rows = len(df)
match_rate_all = df["side_matches"].mean()
neutral_rate = df["is_neutral"].mean()

non_neutral = df[~df["is_neutral"]]
match_rate_non_neutral = non_neutral["side_matches"].mean() if len(non_neutral) else float("nan")

print(f"Rows: {total_rows}")
print(f"Prediction sets leaning to the correct side: {df['side_matches'].sum()} / {total_rows}")
print(f"Match rate including neutral sets: {match_rate_all:.3f}")
print(f"Neutral rate: {neutral_rate:.3f}")
print(f"Match rate excluding neutral sets: {match_rate_non_neutral:.3f}")

# Optional: show a few examples
df[["true_verdict", "prediction_set", "true_side", "prediction_side", "side_matches"]].head(10)

Rows: 55000
Prediction sets leaning to the correct side: 43911 / 55000
Match rate including neutral sets: 0.798
Neutral rate: 0.135
Match rate excluding neutral sets: 0.923


,true_verdict,prediction_set,true_side,prediction_side,side_matches
0,false,['false'],false,false,True
1,false,['false'],false,false,True
2,pants-fire,"['mostly-true', 'half-true', 'mostly-false', '...",false,neutral,False
3,half-true,"['mostly-false', 'false', 'pants-fire']",true,false,False
4,false,"['true', 'mostly-true', 'half-true', 'mostly-f...",false,true,False
5,mostly-false,"['mostly-true', 'half-true', 'mostly-false', '...",false,neutral,False
6,pants-fire,['pants-fire'],false,false,True
7,half-true,"['mostly-true', 'half-true']",true,true,True
8,half-true,"['mostly-true', 'half-true', 'mostly-false', '...",true,neutral,False
9,mostly-true,"['true', 'mostly-true']",true,true,True


crime_path = "C:/Sebi/Master/code/Simple-Test-LLMs-Politifact/results/zero-shot-lac/crime/conformal_results_aggregated_100trials_20260327_141313.csv" 

# for this dataset, make me another that only have the rows where the coverage is 0.0

import pandas as pd
df_crime = pd.read_csv(crime_path)
df_zero_coverage = df_crime[df_crime['coverage_rate'] != 1.0]
df_zero_coverage.to_csv("C:/Sebi/Master/code/Simple-Test-LLMs-Politifact/results/generic/crime/conformal_results_aggregated_100trials_20260327_141313_zero_coverage.csv", index=False)

In [11]:
import pandas as pd

df_claims = pd.read_json("C:/Sebi/Master/code/Simple-Test-LLMs-Politifact/datasets/politifact-crime.json")
df_claims["factcheck_date"] = pd.to_datetime(df_claims["factcheck_date"], errors="coerce")

statement_to_year = (
df_claims.drop_duplicates("statement")
.set_index("statement")["factcheck_date"]
.dt.year
)

df_zero_coverage = df_zero_coverage.copy()
df_zero_coverage["year"] = df_zero_coverage["statement"].map(statement_to_year)

# Order the rows by year
df_zero_coverage = df_zero_coverage.sort_values("year")

df_zero_coverage.to_csv("C:/Sebi/Master/code/Simple-Test-LLMs-Politifact/results/generic/crime/conformal_results_aggregated_100trials_20260327_141313_zero_coverage.csv", index=False)

In [12]:
# For each year, count the number of rows in df_zero_coverage

coverage_by_year = df_zero_coverage.groupby("year").size()
print(coverage_by_year)

year
2007     2
2008     1
2009     1
2010    11
2011    15
2012     4
2013    13
2014    19
2015     9
2016     8
2017    16
2018     8
2019     6
2020     6
2021     3
2022     8
2023     4
2024     5
2025     1
2026     1
dtype: int64


In [13]:
df_crime = df_crime.copy()
df_crime["year"] = df_crime["statement"].map(statement_to_year)

coverage_by_year_all = df_crime.groupby("year").size()
print(coverage_by_year_all)

year
2007     11
2008      7
2009      6
2010     67
2011     54
2012     55
2013     80
2014    102
2015     70
2016    100
2017    104
2018     96
2019     56
2020     83
2021     44
2022     68
2023     29
2024     44
2025     21
2026      3
dtype: int64


In [14]:
# have a percentage of how many rows have zero coverage for each year

coverage_percentage_by_year = coverage_by_year / coverage_by_year_all * 100
print(coverage_percentage_by_year)

year
2007    18.181818
2008    14.285714
2009    16.666667
2010    16.417910
2011    27.777778
2012     7.272727
2013    16.250000
2014    18.627451
2015    12.857143
2016     8.000000
2017    15.384615
2018     8.333333
2019    10.714286
2020     7.228916
2021     6.818182
2022    11.764706
2023    13.793103
2024    11.363636
2025     4.761905
2026    33.333333
dtype: float64
